# Employee Performance Analysis - Prediction
In this notebook we load the saved model and predict the performance rating of an employee from their details.

In [1]:
import pandas as pd
import joblib

model = joblib.load('random_forest_model.pkl')
columns = joblib.load('model_columns.pkl')
encoders = joblib.load('encoders.pkl')

## Entering the Employee Details
Text values are written exactly as they appear in the original data, for example "Sales" or "Yes".

In [2]:
employee = {
    'Age': 32,
    'Gender': 'Male',
    'EducationBackground': 'Life Sciences',
    'MaritalStatus': 'Married',
    'EmpDepartment': 'Sales',
    'EmpJobRole': 'Sales Executive',
    'BusinessTravelFrequency': 'Travel_Rarely',
    'DistanceFromHome': 10,
    'EmpEducationLevel': 3,
    'EmpEnvironmentSatisfaction': 2,
    'EmpHourlyRate': 60,
    'EmpJobInvolvement': 3,
    'EmpJobLevel': 2,
    'EmpJobSatisfaction': 3,
    'NumCompaniesWorked': 2,
    'OverTime': 'Yes',
    'EmpLastSalaryHikePercent': 12,
    'EmpRelationshipSatisfaction': 3,
    'TotalWorkExperienceInYears': 8,
    'TrainingTimesLastYear': 2,
    'EmpWorkLifeBalance': 3,
    'ExperienceYearsAtThisCompany': 5,
    'ExperienceYearsInCurrentRole': 3,
    'YearsSinceLastPromotion': 3,
    'YearsWithCurrManager': 2,
    'Attrition': 'No'
}

In [3]:
new_data = pd.DataFrame([employee])

# convert text values to numbers using the saved encoders
for col in encoders:
    new_data[col] = encoders[col].transform(new_data[col])

# keep the columns in the same order used for training
new_data = new_data[columns]

prediction = model.predict(new_data)
print('Predicted performance rating:', prediction[0])

Predicted performance rating: 2


In [4]:
probabilities = model.predict_proba(new_data)[0]
for rating, prob in zip(model.classes_, probabilities):
    print('Rating', rating, ':', round(prob * 100, 1), '%')

Rating 2 : 81.0 %
Rating 3 : 19.0 %
Rating 4 : 0.0 %


## What-if Test
We change the most important factors for the same employee and check how the prediction changes.

In [5]:
employee_improved = employee.copy()
employee_improved['EmpEnvironmentSatisfaction'] = 4
employee_improved['EmpLastSalaryHikePercent'] = 21
employee_improved['YearsSinceLastPromotion'] = 0

new_data2 = pd.DataFrame([employee_improved])

for col in encoders:
    new_data2[col] = encoders[col].transform(new_data2[col])

new_data2 = new_data2[columns]

print('Predicted performance rating:', model.predict(new_data2)[0])

probabilities2 = model.predict_proba(new_data2)[0]
for rating, prob in zip(model.classes_, probabilities2):
    print('Rating', rating, ':', round(prob * 100, 1), '%')

Predicted performance rating: 4
Rating 2 : 0.0 %
Rating 3 : 35.0 %
Rating 4 : 65.0 %


## Prediction for a Job Candidate
This uses the hiring model, which only needs information known before joining. Its accuracy is about 70%, which is no better than always predicting the most common rating, so it must not be used as the only basis for a hiring decision.

In [6]:
hiring_model = joblib.load('hiring_model.pkl')
hiring_columns = joblib.load('hiring_columns.pkl')

In [7]:
candidate = {
    'Age': 28,
    'EducationBackground': 'Technical Degree',
    'EmpEducationLevel': 3,
    'EmpDepartment': 'Development',
    'EmpJobRole': 'Developer',
    'EmpJobLevel': 2,
    'DistanceFromHome': 8,
    'NumCompaniesWorked': 2,
    'TotalWorkExperienceInYears': 5,
    'EmpHourlyRate': 70,
    'BusinessTravelFrequency': 'Travel_Rarely',
    'OverTime': 'No'
}

In [8]:
candidate_data = pd.DataFrame([candidate])

for col in encoders:
    if col in candidate_data.columns:
        candidate_data[col] = encoders[col].transform(candidate_data[col])

candidate_data = candidate_data[hiring_columns]

print('Predicted performance rating:', hiring_model.predict(candidate_data)[0])

probabilities = hiring_model.predict_proba(candidate_data)[0]
for rating, prob in zip(hiring_model.classes_, probabilities):
    print('Rating', rating, ':', round(prob * 100, 1), '%')

Predicted performance rating: 3
Rating 2 : 1.0 %
Rating 3 : 84.0 %
Rating 4 : 15.0 %
